# 🏪 Ecommerce Agent: Skills or Subagents?

**After-session learner exercise.** Build both architectures, run
the same mixed Product+Order request, and compare their Phoenix traces.

> **Tracing notice:** Phoenix receives prompts, model activity, tool
arguments, and synthetic results. Never enter real customer data.

## Goal

- **Part A:** one parent with Product and Order specialist subagents.
- **Part B:** one agent with Product and Order Skills.

There is no universal winner. Use the traces to decide where each
boundary is helpful.

## Setup

### Connect the model and Phoenix

Hosted JupyterLab already contains the dependencies from
notebooks/requirements.txt. Run the one collapsed helper cell ▶️.
The prepared library handles the environment, model connection, and
Phoenix tracing. Your JupyterHub username becomes the Phoenix project
name. Use only synthetic classroom examples.

In [ ]:
from google.adk.skills import load_skill_from_dir
from google.adk.tools.skill_toolset import SkillToolset

from mcp_workshop.workshop_helpers import (
    Agent,
    GOOGLE_MODEL,
    close_workshop_tools,
    load_order_mcp_tools,
    load_order_skill,
    load_order_skill_tools,
    load_product_mcp_tools,
    make_order_agent,
    make_workshop_runner,
    repository_root,
    run_traced_turn,
    tracer_provider,
)

PRODUCT_SKILL_DIR = (
    repository_root / "notebooks" / "product_takehome" / "product-shopping-code"
)
ECOMMERCE_PROMPT = "Find two highly rated 55-inch 4K TVs under $400. Retrieve the full product details for both, compare availability, seller, brand, and key specifications, then recommend one. Also find my most recent order from July 1 through July 31, 2026. Tell me its status, total, and every item's name and quantity. Clearly identify any missing information."

## Steps

### 🧑‍🤝‍🧑 Part A — specialist subagents

The Order specialist is provided and already uses `mode="single_turn"`
with the two direct Order MCP tools. Confirm it, then build the matching
Product specialist from your Notebook 07 work.

In [ ]:
order_agent = make_order_agent()
assert order_agent.name == "order_agent"
assert order_agent.mode == "single_turn"
print("📦 Provided Order specialist ready.")

#### ✍️ Product specialist instructions

Copy or adapt your grounded direct-MCP instructions from Notebook 07.

In [ ]:
PRODUCT_INSTRUCTIONS = ""  # YOUR TURN
assert PRODUCT_INSTRUCTIONS.strip(), "Add Product specialist instructions."

#### 🤖 Build the Product specialist

Keep it single-turn so the ecommerce parent can delegate one bounded job.

In [ ]:
product_mcp_tools = load_product_mcp_tools()
product_agent = Agent(
    name="",  # YOUR TURN: product_agent
    model=GOOGLE_MODEL,
    mode="single_turn",
    instruction=PRODUCT_INSTRUCTIONS,
    tools=[],  # YOUR TURN: add product_mcp_tools
)
assert product_agent.name == "product_agent"
assert product_agent.tools == [product_mcp_tools]

#### ✍️ Parent instructions

Tell the parent to delegate Order and Product work to the matching
specialist and combine only grounded returned results.

In [ ]:
ECOMMERCE_SUBAGENT_INSTRUCTIONS = ""  # YOUR TURN
assert ECOMMERCE_SUBAGENT_INSTRUCTIONS.strip(), "Add parent instructions."

#### 🏗️ Build the parent ecommerce agent

The parent owns the conversation; specialists own their tools and loops.

In [ ]:
ecommerce_subagent_agent = Agent(
    name="",  # YOUR TURN: ecommerce_agent
    model=GOOGLE_MODEL,
    instruction=ECOMMERCE_SUBAGENT_INSTRUCTIONS,
    sub_agents=[],  # YOUR TURN: add order_agent and product_agent
)
assert ecommerce_subagent_agent.name == "ecommerce_agent"
assert ecommerce_subagent_agent.sub_agents == [order_agent, product_agent]
ecommerce_subagent_runner = make_workshop_runner(
    ecommerce_subagent_agent,
    "08-ecommerce-subagents",
)

#### ▶️ Run Part A

Keep this prepared mixed request unchanged for both architectures.

In [ ]:
_ = await run_traced_turn(
    trace_name="08-A-ecommerce-subagent-agent",
    runner=ecommerce_subagent_runner,
    prompt=ECOMMERCE_PROMPT,
    session_id="08-A-ecommerce-subagent-agent",
)

### 📚 Part B — one agent with two Skills

The complete Order Skill and Order MCP wiring are provided. Finish the
Product Skill from Notebook 06, then place both procedures inside one
agent boundary.

In [ ]:
order_skill = load_order_skill()
order_skill_tools = load_order_skill_tools()
order_mcp_tools = load_order_mcp_tools()
assert order_skill.frontmatter.name == "order-support"
print("📦 Provided Order Skill and tools ready.")

#### 📘 Finish and load your Product Skill

Open `PRODUCT_SKILL_DIR / "SKILL.md"`, replace all five `YOUR TURN`
lines, and rerun this check.

In [ ]:
product_skill_source = (
    PRODUCT_SKILL_DIR / "SKILL.md"
).read_text(encoding="utf-8")
assert "YOUR TURN" not in product_skill_source, "Finish the Product Skill."
product_skill = load_skill_from_dir(PRODUCT_SKILL_DIR)
product_mcp_tools = load_product_mcp_tools()
print("🛍️ Product Skill ready.")

#### ✍️ One-agent instructions

Explain when to use each Skill and require grounded synthesis. Keep
domain procedures in the Skills instead of duplicating them here.

In [ ]:
ECOMMERCE_SKILL_INSTRUCTIONS = ""  # YOUR TURN
assert ECOMMERCE_SKILL_INSTRUCTIONS.strip(), "Add ecommerce Skill instructions."

#### 🏗️ Build one ecommerce agent with both Skills

This architecture shares one prompt, one conversation context, and
one model loop. Skill activation controls which procedure and declared
tools enter the active work.

In [ ]:
ECOMMERCE_SKILLS = []  # YOUR TURN: add order_skill and product_skill
ECOMMERCE_ADDITIONAL_TOOLS = []  # YOUR TURN: add both MCP toolsets
assert ECOMMERCE_SKILLS == [order_skill, product_skill]
assert ECOMMERCE_ADDITIONAL_TOOLS == [order_mcp_tools, product_mcp_tools]
ecommerce_skill_toolset = SkillToolset(
    skills=ECOMMERCE_SKILLS,
    additional_tools=ECOMMERCE_ADDITIONAL_TOOLS,
)
ecommerce_skill_agent = Agent(
    name="",  # YOUR TURN: ecommerce_agent
    model=GOOGLE_MODEL,
    instruction=ECOMMERCE_SKILL_INSTRUCTIONS,
    tools=[],  # YOUR TURN: add ecommerce_skill_toolset
)
assert ecommerce_skill_agent.name == "ecommerce_agent"
assert ecommerce_skill_agent.tools == [ecommerce_skill_toolset]
ecommerce_skill_runner = make_workshop_runner(
    ecommerce_skill_agent,
    "08-ecommerce-skills",
)

#### ▶️ Run Part B

Use the identical Product+Order request.

In [ ]:
_ = await run_traced_turn(
    trace_name="08-B-ecommerce-skill-agent",
    runner=ecommerce_skill_runner,
    prompt=ECOMMERCE_PROMPT,
    session_id="08-B-ecommerce-skill-agent",
)

## Checks

### 🔎 Compare the two Phoenix traces

Write a short answer under every question.

1. **How many model calls occurred in each architecture?**
   _Your answer:_
2. **Where did delegation appear in the subagent trace?**
   _Your answer:_
3. **Which instructions and tools entered each context?**
   _Your answer:_
4. **Which architecture used more input and output tokens?**
   _Your answer:_
5. **Did Product context enter Order specialist work, or vice versa?**
   _Your answer:_
6. **Did the parent combine grounded results without inventing fields?**
   _Your answer:_
7. **Which architecture would the learner choose for a small stable agent?**
   _Your answer:_
8. **Which architecture would the learner choose as more domains and domain-specific policies are added?**
   _Your answer:_

### 🧹 Clean up

Close all shared MCP connections after both traces are flushed.

In [ ]:
await close_workshop_tools()
print("🧹 Ecommerce workshop connections closed.")

## Next Steps

Prefer one agent with Skills when shared context stays safe and simple.
Prefer specialist subagents when domains need isolated prompts, tools,
context, policies, or independent reasoning loops.